# Visualização da Informação — Pedidos de E-commerce (2023–2025)

**Dataset real:** *E-commerce Orders and Customer Analytics Dataset* (Kaggle, licença PDDL), 1.200 pedidos de 01/01/2023 a 30/06/2025, 14 colunas. Link do dataset: https://www.kaggle.com/datasets/hammadansari7/e-commerce-orders-and-customer-analytics-dataset (arquivo `data/ecommerce/ecommerce_orders_dataset.csv.xlsx`).

Três gráficos de **unidades diferentes** da disciplina, todos sobre o mesmo dataset e encadeados em uma única pergunta: *de onde vem a receita da loja, como ela evolui no tempo e quanto dela de fato se concretiza?*

| Unidade | Técnica | Papel na narrativa |
|---|---|---|
| Estatística descritiva | Barras | Receita total por produto |
| Temporal | Linhas | Receita mensal e tendência |
| Hierárquica | Sunburst | Status do pedido → forma de pagamento → produto |

> "Receita" aqui é a soma de `TotalPrice` de **todos** os pedidos (inclui cancelados e devolvidos). O dataset não informa a moeda.

In [ ]:
import pandas as pd
import plotly.express as px

# O arquivo se chama .csv.xlsx, mas é uma planilha Excel
df = pd.read_excel('data/ecommerce/ecommerce_orders_dataset.csv.xlsx')

# Tradução dos rótulos categóricos para o português dos gráficos
status_pt = {'Delivered': 'Entregue', 'Shipped': 'Enviado', 'Pending': 'Pendente',
             'Cancelled': 'Cancelado', 'Returned': 'Devolvido'}
pagto_pt = {'Online': 'Online', 'Cash': 'Dinheiro', 'Credit Card': 'Cartão de crédito',
            'Debit Card': 'Cartão de débito', 'Gift Card': 'Vale-presente'}
produto_pt = {'Printer': 'Impressora', 'Tablet': 'Tablet', 'Chair': 'Cadeira', 'Laptop': 'Notebook',
              'Desk': 'Escrivaninha', 'Monitor': 'Monitor', 'Phone': 'Celular'}
df['status'] = df['OrderStatus'].map(status_pt)
df['pagamento'] = df['PaymentMethod'].map(pagto_pt)
df['produto'] = df['Product'].map(produto_pt)

print(df.shape, df['Date'].min().date(), '->', df['Date'].max().date())
print('Pedidos duplicados:', int(df['OrderID'].duplicated().sum()))
print('Nulos em TotalPrice:', int(df['TotalPrice'].isna().sum()))
df[['OrderID', 'Date', 'produto', 'Quantity', 'UnitPrice', 'pagamento', 'status', 'TotalPrice']].head()

## Plot 1 — Barras (Estatística descritiva)
A receita total é uma grandeza somada por categoria discreta (produto), a natureza ideal para barras: comprimento comparável a partir de uma base comum, ordenado do maior para o menor.

In [ ]:
por_produto = (df.groupby('produto')['TotalPrice']
               .agg(receita='sum', pedidos='count', ticket_medio='mean')
               .reset_index().sort_values('receita', ascending=False))

fig1 = px.bar(
    por_produto, x='produto', y='receita', text=por_produto['receita'].round(0),
    hover_data={'pedidos': True, 'ticket_medio': ':.2f', 'receita': ':,.2f'},
    labels={'produto': 'Produto', 'receita': 'Receita total', 'pedidos': 'Pedidos',
            'ticket_medio': 'Ticket médio'},
    title='Receita total por produto (jan/2023 a jun/2025)',
)
fig1.update_traces(marker_color='#2a6f97', textposition='outside', texttemplate='%{text:,.0f}')
fig1.update_layout(yaxis_range=[0, por_produto['receita'].max() * 1.15])
fig1.show()

## Plot 2 — Linhas (Temporal)
Série temporal contínua: a linha preserva a ordem dos meses. A média móvel de 3 meses separa a tendência da oscilação mensal.

In [ ]:
mensal = (df.set_index('Date')['TotalPrice'].resample('MS').sum()
          .rename('receita').reset_index())
mensal['media_movel_3m'] = mensal['receita'].rolling(3, min_periods=1).mean()

fig2 = px.line(
    mensal, x='Date', y=['receita', 'media_movel_3m'], markers=True,
    labels={'Date': 'Mês', 'value': 'Receita', 'variable': 'Série'},
    title='Receita mensal e média móvel de 3 meses',
)
fig2.data[0].name = 'Receita mensal'
fig2.data[0].line.update(color='#a9c6dc', width=1.5)
fig2.data[1].name = 'Média móvel 3 meses'
fig2.data[1].line.update(color='#c1121f', width=3)
fig2.data[1].marker.update(size=0)
fig2.update_xaxes(rangeslider_visible=True)
fig2.show()

## Plot 3 — Sunburst (Hierárquica)
Cada pedido pertence a um status, foi pago de uma forma e é de um produto: status → forma de pagamento → produto. O ângulo de cada fatia é a receita; a cor, o ticket médio. Mostra quanto da receita se concretiza (entregue/enviado) e quanto se perde (cancelado/devolvido).

In [ ]:
fig3 = px.sunburst(
    df, path=['status', 'pagamento', 'produto'], values='TotalPrice',
    color='TotalPrice', color_continuous_scale='YlOrRd',
    labels={'TotalPrice': 'Ticket médio'},
    title='Receita por status > forma de pagamento > produto (cor = ticket médio)',
)
fig3.update_layout(width=750, height=750)
fig3.show()

## Conclusão
- As **barras** mostram uma receita total de 1,26 milhão distribuída de forma relativamente equilibrada: Cadeira (195,6 mil) e Impressora (195,6 mil) empatam na liderança, e Celular fica por último (151,7 mil), com o menor ticket médio (972,58).
- As **linhas** mostram a oscilação mensal: o pico é junho/2024 (68,1 mil) e o vale é abril/2023 (27,8 mil). Os primeiros semestres de 2023 e 2024 foram mais fortes que o de 2025.
- O **sunburst** mostra que só 38,6% da receita é de pedidos entregues ou enviados; **41,1%** está em pedidos cancelados (21,9%) ou devolvidos (19,2%), e o restante está pendente (20,3%).

_Os valores de receita foram calculados sobre `TotalPrice` e a moeda não é informada pelo dataset. Revisar os comentários acima contra os gráficos gerados antes de gravar o vídeo._